In [1]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score
import xgboost as xgb
import lightgbm as lgb

print("All imports done!")

All imports done!


In [2]:
# Cell 2 - Load data from GitHub
url = "https://raw.githubusercontent.com/Shar1q-codes/factoryguard-ai/master/data/train_FD001_with_RUL.csv"

df = pd.read_csv(url)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head(2))

HTTPError: HTTP Error 404: Not Found

In [3]:
# Cell 2 - Load data from GitHub
url = "https://raw.githubusercontent.com/Shar1q-codes/factoryguard-ai/master/data/train_FD001_with_RUL.csv"

df = pd.read_csv(url)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head(2))

HTTPError: HTTP Error 404: Not Found

In [4]:
from google.colab import files
uploaded = files.upload()

Saving train_FD001_with_RUL.csv to train_FD001_with_RUL.csv


In [5]:
df = pd.read_csv('train_FD001_with_RUL.csv')
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head(2))

Shape: (24640, 29)
Columns: ['unit_nr', 'time_cycles', 'op_setting_1', 'op_setting_2', 'op_setting_3', 's1', 's2', 's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's20', 's21', 'max_cycles', 'RUL', 'failure']
   unit_nr  time_cycles  op_setting_1  op_setting_2  op_setting_3       s1  \
0        1            1        0.3949        0.2935       60.5632  83.9352   
1        1            2        0.4038        0.0649       70.1566  35.8033   

        s2       s3       s4       s5  ...      s15      s16      s17  \
0  56.3191  56.9230  24.0660  95.0557  ...  65.2907  52.2463  78.5608   
1  83.3367  93.5509  56.6263  72.7263  ...  53.2672  46.2497  41.6661   

       s18      s19      s20      s21  max_cycles  RUL  failure  
0  98.2491  80.0192  68.5879  43.6535         252  251        0  
1  19.5831  62.4714  75.0253  37.2535         252  250        0  

[2 rows x 29 columns]


In [6]:
# Cell 3 - Load data
df = pd.read_csv('train_FD001_with_RUL.csv')
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head(2))

Shape: (24640, 29)
Columns: ['unit_nr', 'time_cycles', 'op_setting_1', 'op_setting_2', 'op_setting_3', 's1', 's2', 's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's20', 's21', 'max_cycles', 'RUL', 'failure']
   unit_nr  time_cycles  op_setting_1  op_setting_2  op_setting_3       s1  \
0        1            1        0.3949        0.2935       60.5632  83.9352   
1        1            2        0.4038        0.0649       70.1566  35.8033   

        s2       s3       s4       s5  ...      s15      s16      s17  \
0  56.3191  56.9230  24.0660  95.0557  ...  65.2907  52.2463  78.5608   
1  83.3367  93.5509  56.6263  72.7263  ...  53.2672  46.2497  41.6661   

       s18      s19      s20      s21  max_cycles  RUL  failure  
0  98.2491  80.0192  68.5879  43.6535         252  251        0  
1  19.5831  62.4714  75.0253  37.2535         252  250        0  

[2 rows x 29 columns]


In [7]:
# Cell 4 - Check columns & define sensors
sensors = [f's{i}' for i in range(1, 22)]
print("Sensors:", sensors)
print("Total sensors:", len(sensors))

Sensors: ['s1', 's2', 's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's20', 's21']
Total sensors: 21


In [8]:
# Cell 5 - Rolling Features
def add_rolling_features(df, sensors, windows=[5, 10, 20]):
    for col in sensors:
        for w in windows:
            df[f'{col}_roll_mean_{w}'] = df.groupby('unit_nr')[col].transform(
                lambda x: x.rolling(w, min_periods=1).mean()
            )
            df[f'{col}_roll_std_{w}'] = df.groupby('unit_nr')[col].transform(
                lambda x: x.rolling(w, min_periods=1).std().fillna(0)
            )
            df[f'{col}_ema_{w}'] = df.groupby('unit_nr')[col].transform(
                lambda x: x.ewm(span=w, adjust=False).mean()
            )
    return df

df = add_rolling_features(df, sensors)
print("Rolling features done! Shape:", df.shape)

/tmp/ipykernel_6752/306300385.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_ema_{w}'] = df.groupby('unit_nr')[col].transform(
/tmp/ipykernel_6752/306300385.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_roll_mean_{w}'] = df.groupby('unit_nr')[col].transform(
/tmp/ipykernel_6752/306300385.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=

Rolling features done! Shape: (24640, 218)


/tmp/ipykernel_6752/306300385.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_roll_std_{w}'] = df.groupby('unit_nr')[col].transform(
/tmp/ipykernel_6752/306300385.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_ema_{w}'] = df.groupby('unit_nr')[col].transform(
/tmp/ipykernel_6752/306300385.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1

In [9]:
# Cell 6 - Fix fragmentation + check shape
df = df.copy()
print("Rolling features done! Shape:", df.shape)

Rolling features done! Shape: (24640, 218)


In [10]:
# Cell 7 - Lag Features
def add_lag_features(df, sensors):
    for col in sensors:
        df[f'{col}_lag1'] = df.groupby('unit_nr')[col].transform(
            lambda x: x.shift(1)
        ).fillna(0)
        df[f'{col}_lag2'] = df.groupby('unit_nr')[col].transform(
            lambda x: x.shift(2)
        ).fillna(0)
    return df

df = add_lag_features(df, sensors)
df = df.copy()
print("Lag features done! Shape:", df.shape)

Lag features done! Shape: (24640, 260)


In [11]:
# Cell 8 - Train/Test Split
drop_cols = ['unit_nr', 'time_cycles', 'max_cycles', 'RUL']

units = df['unit_nr'].unique()
train_units = units[:int(len(units)*0.8)]
test_units = units[int(len(units)*0.8):]

train = df[df['unit_nr'].isin(train_units)]
test = df[df['unit_nr'].isin(test_units)]

X_train = train.drop(columns=drop_cols)
y_train = train['failure']
X_test = test.drop(columns=drop_cols)
y_test = test['failure']

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Failure ratio train:", y_train.mean().round(4))

X_train shape: (19468, 256)
X_test shape: (5172, 256)
Failure ratio train: 0.1274


In [12]:
# Cell 9 - XGBoost
ratio = (y_train == 0).sum() / (y_train == 1).sum()
print("Scale pos weight:", ratio.round(2))

xgb_model = xgb.XGBClassifier(
    scale_pos_weight=ratio,
    n_estimators=200,
    use_label_encoder=False,
    eval_metric='aucpr'
)
xgb_model.fit(X_train, y_train)
xgb_prauc = average_precision_score(y_test, xgb_model.predict_proba(X_test)[:,1])
print(f"XGBoost PR-AUC: {xgb_prauc:.4f}")

Scale pos weight: 6.85


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [05:55:31] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost PR-AUC: 1.0000


In [13]:
# Cell 10 - LightGBM
lgb_model = lgb.LGBMClassifier(
    scale_pos_weight=ratio,
    n_estimators=200
)
lgb_model.fit(X_train, y_train)
lgb_prauc = average_precision_score(y_test, lgb_model.predict_proba(X_test)[:,1])
print(f"LightGBM PR-AUC: {lgb_prauc:.4f}")

[LightGBM] [Info] Number of positive: 2480, number of negative: 16988
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.048517 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 65027
[LightGBM] [Info] Number of data points in the train set: 19468, number of used features: 256
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.127389 -> initscore=-1.924249
[LightGBM] [Info] Start training from score -1.924249
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

In [14]:
# Cell 11 - Compare & Save
print("="*40)
print(f"XGBoost PR-AUC:  {xgb_prauc:.4f}")
print(f"LightGBM PR-AUC: {lgb_prauc:.4f}")
print("="*40)
print("Dono barabar hain - XGBoost recommend karo Chemala ko!")
print("(XGBoost faster hai tune karne ke liye Optuna se)")

# Save features
joblib.dump(df, 'features_df.pkl')
print("features_df.pkl saved!")

XGBoost PR-AUC:  1.0000
LightGBM PR-AUC: 1.0000
Dono barabar hain - XGBoost recommend karo Chemala ko!
(XGBoost faster hai tune karne ke liye Optuna se)
features_df.pkl saved!


In [15]:
from google.colab import files
files.download('features_df.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>